In [1]:
import numpy as np
import plotly.graph_objects as go

# 1. Definición de la superficie de pérdida en R^3 y su gradiente
def loss(x, y, z):
    return x**2 + 2*y**2 + 3*z**2 - 2*x*y - 2*y*z

def grad_loss(x, y, z):
    gx = 2*x - 2*y
    gy = 4*y - 2*x - 2*z
    gz = 6*z - 2*y
    return np.array([gx, gy, gz])

# 2. Trayectoria 1: Descenso por Coordenadas (Optimización 1 a 1 en pasos a 90°)
path_cd = [(4.0, 4.0, 4.0)]
x_c, y_c, z_c = 4.0, 4.0, 4.0

for _ in range(8):
    # Pasos analíticos resolviendo la derivada parcial = 0 para cada coordenada
    x_c = y_c                          # dL/dx = 0 -> 2x - 2y = 0
    path_cd.append((x_c, y_c, z_c))

    y_c = (x_c + z_c) / 2.0             # dL/dy = 0 -> 4y - 2x - 2z = 0
    path_cd.append((x_c, y_c, z_c))

    z_c = y_c / 3.0                    # dL/dz = 0 -> 6z - 2y = 0
    path_cd.append((x_c, y_c, z_c))

path_cd = np.array(path_cd)

# 3. Trayectoria 2: Descenso de Gradiente Conjunto (Vectorial directo)
path_gd = [(4.0, 4.0, 4.0)]
curr_pos = np.array([4.0, 4.0, 4.0])
eta = 0.15  # Learning rate

for _ in range(25):
    g = grad_loss(*curr_pos)
    curr_pos = curr_pos - eta * g
    path_gd.append(tuple(curr_pos))

path_gd = np.array(path_gd)

# 4. Generación de las isosuperficies (superficies de nivel de pérdida constante en R^3)
grid_range = np.linspace(-0.5, 4.5, 30)
X, Y, Z = np.meshgrid(grid_range, grid_range, grid_range)
VALUES = loss(X, Y, Z)

# 5. Construcción de la figura interactiva 3D con Plotly
fig = go.Figure()

# Isosuperficies (Elipsoides rotados)
fig.add_trace(go.Isosurface(
    x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
    value=VALUES.flatten(),
    isomin=2.0, isomax=25.0,
    surface_count=4,
    opacity=0.25,
    colorscale='Viridis',
    showscale=False,
    name='Superficies de Nivel L(x,y,z)'
))

# Trayectoria 1: Pasos por Coordenadas (Rojo - Ángulos rectos)
fig.add_trace(go.Scatter3d(
    x=path_cd[:, 0], y=path_cd[:, 1], z=path_cd[:, 2],
    mode='lines+markers',
    line=dict(color='red', width=5),
    marker=dict(size=4, color='darkred'),
    name='Por Coordenadas (Pasos a 90°)'
))

# Trayectoria 2: Gradiente Conjunto (Azul - Vectorial directo)
fig.add_trace(go.Scatter3d(
    x=path_gd[:, 0], y=path_gd[:, 1], z=path_gd[:, 2],
    mode='lines+markers',
    line=dict(color='blue', width=6, dash='dash'),
    marker=dict(size=4, color='darkblue'),
    name='Gradiente Conjunto (Directo)'
))

# Marcar Inicio y Mínimo Absoluto
fig.add_trace(go.Scatter3d(
    x=[4.0], y=[4.0], z=[4.0],
    mode='markers',
    marker=dict(size=8, color='green', symbol='circle'),
    name='Inicio (4, 4, 4)'
))

fig.add_trace(go.Scatter3d(
    x=[0.0], y=[0.0], z=[0.0],
    mode='markers',
    marker=dict(size=10, color='gold', symbol='diamond'),
    name='Mínimo global (0, 0, 0)'
))

# Configuración del Layout e Interactividad
fig.update_layout(
    title=dict(
        text=r'<b>Optimización en R³: L(x, y, z) = x² + 2y² + 3z² - 2xy - 2yz</b>',
        x=0.5, font=dict(size=14)
    ),
    scene=dict(
        xaxis_title='Parámetro X',
        yaxis_title='Parámetro Y',
        zaxis_title='Parámetro Z',
        aspectmode='cube'
    ),
    width=900,
    height=700,
    margin=dict(r=10, l=10, b=10, t=40)
)

fig.show()

In [2]:
import numpy as np
import plotly.graph_objects as go

# 1. Función de pérdida acoplada L(x, y) = x^2 + 4y^2 - 3xy
def loss(x, y):
    return x**2 + 4*y**2 - 3*x*y

def grad_x(x, y):
    return 2*x - 3*y

def grad_y(x, y):
    return 8*y - 3*x

# 2. Trayectoria por Coordenadas (Optimización 1 a 1: X luego Y)
path_cd = [(4.0, 4.0, loss(4.0, 4.0))]
x_c, y_c = 4.0, 4.0

for _ in range(6):
    # Optimizar X fijando Y (dL/dx = 0 -> x = 1.5 * y)
    x_c = 1.5 * y_c
    path_cd.append((x_c, y_c, loss(x_c, y_c)))

    # Optimizar Y fijando X (dL/dy = 0 -> y = 3/8 * x)
    y_c = (3.0 / 8.0) * x_c
    path_cd.append((x_c, y_c, loss(x_c, y_c)))

path_cd = np.array(path_cd)

# 3. Trayectoria por Gradiente Conjunto (Vectorial directo)
path_gd = [(4.0, 4.0, loss(4.0, 4.0))]
x_g, y_g = 4.0, 4.0
eta = 0.12  # Learning rate

for _ in range(12):
    gx = grad_x(x_g, y_g)
    gy = grad_y(x_g, y_g)
    x_g -= eta * gx
    y_g -= eta * gy
    path_gd.append((x_g, y_g, loss(x_g, y_g)))

path_gd = np.array(path_gd)

# 4. Malla para renderizar la superficie Z = L(x, y)
x_vals = np.linspace(-1, 6.5, 100)
y_vals = np.linspace(-1, 4.5, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = loss(X, Y)

# 5. Construcción de la figura 3D
fig = go.Figure()

# Superficie continua Z = L(x, y)
fig.add_trace(go.Surface(
    x=X, y=Y, z=Z,
    colorscale='Viridis',
    opacity=0.75,
    showscale=False,
    contours_z=dict(show=True, usecolormap=True, highlightcolor="limegreen", project_z=True),
    name='Superficie L(x, y)'
))

# Trayectoria por Coordenadas (Rojo - Pasos a 90° sobre la superficie)
fig.add_trace(go.Scatter3d(
    x=path_cd[:, 0], y=path_cd[:, 1], z=path_cd[:, 2],
    mode='lines+markers',
    line=dict(color='red', width=6),
    marker=dict(size=4, color='darkred'),
    name='Por Coordenadas (Pasos a 90°)'
))

# Trayectoria por Gradiente Conjunto (Azul - Descenso directo)
fig.add_trace(go.Scatter3d(
    x=path_gd[:, 0], y=path_gd[:, 1], z=path_gd[:, 2],
    mode='lines+markers',
    line=dict(color='blue', width=6, dash='dash'),
    marker=dict(size=4, color='darkblue'),
    name='Gradiente Conjunto'
))

# Puntos de Inicio y Mínimo Global
fig.add_trace(go.Scatter3d(
    x=[4.0], y=[4.0], z=[loss(4.0, 4.0)],
    mode='markers',
    marker=dict(size=7, color='green'),
    name='Inicio (4, 4)'
))

fig.add_trace(go.Scatter3d(
    x=[0.0], y=[0.0], z=[0.0],
    mode='markers',
    marker=dict(size=9, color='gold', symbol='diamond'),
    name='Mínimo (0, 0)'
))

# Configuración del escenario
fig.update_layout(
    title=dict(
        text=r'<b>Superficie de Pérdida 3D: L(x, y) = x² + 4y² - 3xy</b>',
        x=0.5, font=dict(size=14)
    ),
    scene=dict(
        xaxis_title='Parámetro X',
        yaxis_title='Parámetro Y',
        zaxis_title='Pérdida L(x, y)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2)
        )
    ),
    width=900,
    height=700,
    margin=dict(r=10, l=10, b=10, t=40)
)

fig.show()